# CrossFuse v3 — Two-Head (Video / Audio) Deepfake Detector

This notebook detects deepfakes on **FakeAVCeleb**, whose clips come in four flavours: RealVideo-RealAudio (RVRA), RealVideo-FakeAudio (RVFA), FakeVideo-RealAudio (FVRA), FakeVideo-FakeAudio (FVFA). Instead of collapsing all of that into one real/fake score (v1/v2, which returned ~50% on uploaded clips), **v3 predicts the two tracks separately**:

- a **video head** reading only the visual branch → `P(video is fake)`
- an **audio head** reading only the audio branch → `P(audio is fake)`

A clip is called **legit only if both heads say real**; otherwise the verdict names which track (or both) is manipulated. Cross-attention is retained purely as an auxiliary "any-manipulation" regularizer and never feeds the two attribution heads, so the two decisions stay independent.

**Cell-by-cell summary:**
- **Cell 1** — env setup; adds `USE_MFCC_CMVN` and a shared `cmvn_normalize()` audio-normalization helper used identically in Cells 3/6/7
- **Cell 2** — model rewritten into two separate branches + two binary heads; `predict_with_uncertainty()` now returns per-head `(mean_v, std_v, mean_a, std_a)`
- **Cell 3** — each sample stores `video_label` + `audio_label`; audio-desync negatives dropped; self-blend → `video_fake` only; category-stratified fake selection so the audio head gets positives; MFCC CMVN
- **Cell 4** — identity-disjoint split; multi-label `BCE(video) + BCE(audio)`; separate val AUC per head
- **Cell 5** — per-head calibration (two thresholds), per-head ECE + reliability, and a 4-way attribution confusion matrix
- **Cell 6** — DFDC zero-shot evaluated on the **video head** (DFDC is a video benchmark); both prior DFDC bug fixes retained
- **Cell 7** — interactive inference reporting a **per-modality** verdict with per-head uncertainty

**Run cells in order.** ⚠️ **Before rerunning after these changes, delete the cached feature folders** (`/kaggle/working/extracted_features` and `/kaggle/working/dfdc_test_features`) — Cell 3/6 skip any `.pt` that already exists, so stale tensors from the old binary-label schema would otherwise be reused silently. Read the `WHAT CHANGED` comment at the top of each code cell before running it.

### Cell 1 — Environment Setup (mostly unchanged)
Adds `GroupShuffleSplit` (needed for Cell 4's identity-disjoint split) and three config flags you can toggle without touching any other cell:
- `GENERATE_HARD_NEGATIVES` — set `False` to fall back to v1's dataset (faster, but skips the generalization fix)
- `HELD_OUT_CATEGORY` — optional: name one FakeAVCeleb category (e.g. `"FakeVideo-RealAudio"`) to hold out entirely, for an optional cross-manipulation ablation row in Table 4
- `N_MC_SAMPLES` — how many stochastic forward passes MC-Dropout uses per prediction (20 is a reasonable speed/quality tradeoff)


In [3]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP & SAFE DEPENDENCY LOADING  (v2)
# NOTE: This version installs the extra packages with --no-deps so they
#       CANNOT upgrade or downgrade numpy/torch/torchvision as a side
#       effect (that side-effect upgrade was the cause of the repeated
#       "numpy.dtype size changed" crash). Kaggle's stock numpy/torch/
#       torchvision are already a matched set -- we leave them untouched.
# =====================================================================
import os
import sys
import glob
import json
import random
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

print("Installing extra dependencies WITHOUT touching numpy/torch/torchvision...")
# --no-deps => pip installs ONLY these packages' own code and resolves
# nothing else, so it is impossible for numpy to be changed here.
!pip install -q --no-deps facenet-pytorch librosa ffmpeg-python tqdm soundfile audioread pooch lazy_loader
print("Installation complete!")

import PIL._util
if not hasattr(PIL._util, 'is_directory'):
    PIL._util.is_directory = os.path.isdir
if not hasattr(PIL._util, 'is_path'):
    PIL._util.is_path = lambda x: isinstance(x, (str, bytes, os.PathLike))

import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                              roc_curve, auc, classification_report)
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
import seaborn as sns

print(f"\nVersion check -> numpy: {np.__version__} | torch: {torch.__version__} | torchvision: {torchvision.__version__ if 'torchvision' in dir() else 'imported'}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(seed=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Environment ready! Hardware acceleration running on: {device.type.upper()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# ---- Shared MFCC normalization (CMVN) --------------------------------
# Per-utterance Cepstral Mean-Variance Normalization: standardize each of
# the 40 MFCC coefficients across the time axis so an uploaded phone clip's
# audio lands in the SAME distribution the model trained on. This single
# helper is used identically in Cells 3 (train features), 6 (DFDC) and 7
# (live inference) -- keep it here so it is defined before all of them.
def cmvn_normalize(mfcc, eps=1e-8):
    mfcc = np.asarray(mfcc, dtype=np.float32)
    mean = mfcc.mean(axis=1, keepdims=True)
    std = mfcc.std(axis=1, keepdims=True)
    return (mfcc - mean) / (std + eps)

GENERATE_HARD_NEGATIVES = True
HELD_OUT_CATEGORY = None
N_MC_SAMPLES = 20
USE_MFCC_CMVN = True   # per-utterance MFCC normalization (Cells 3/6/7 all honor this)
print(f"Config -> GENERATE_HARD_NEGATIVES={GENERATE_HARD_NEGATIVES} | HELD_OUT_CATEGORY={HELD_OUT_CATEGORY} "
      f"| N_MC_SAMPLES={N_MC_SAMPLES} | USE_MFCC_CMVN={USE_MFCC_CMVN}")

Installing extra dependencies WITHOUT touching numpy/torch/torchvision...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.0 MB/s eta 0:00:0000:010:01
Installation complete!

Version check -> numpy: 2.0.2 | torch: 2.10.0+cu128 | torchvision: imported
Environment ready! Hardware acceleration running on: CUDA (Tesla T4)
Config -> GENERATE_HARD_NEGATIVES=True | HELD_OUT_CATEGORY=None | N_MC_SAMPLES=20 | USE_MFCC_CMVN=True


### Cell 2 — Two-Head Model (Separate Branches) + MC-Dropout Uncertainty
**v3 change:** the single binary real/fake head is replaced by **two independent binary heads**, matching FakeAVCeleb's 2×2 structure:
- **`video_head`** reads the **visual branch only** → `P(video is fake)`
- **`audio_head`** reads the **audio branch only** → `P(audio is fake)`

Because each head sees only its own modality, the model can *attribute* which track is manipulated instead of just saying "fake." The cross-attention path is kept purely as an **auxiliary "any-manipulation" regularizer** (`any_logit`) that never feeds the two attribution heads — so audio can't leak into the video decision or vice-versa. The final verdict (Cell 7) is **legit iff both heads say real**.

`predict_with_uncertainty()` now returns **four** arrays — `(mean_v, std_v, mean_a, std_a)` — one mean/uncertainty pair per head. `forward()` returns `(video_logit, audio_logit, any_logit, attn_weights)` as **raw logits** (train with `BCEWithLogitsLoss`, read with `torch.sigmoid`).

In [4]:
# =====================================================================
# CELL 2: CROSSFUSE ARCHITECTURE (TWO-HEAD, SEPARATE BRANCHES) +
#         MC-DROPOUT UNCERTAINTY UTILITIES  (v3)
# WHAT CHANGED vs v2: the single binary real/fake head is replaced by TWO
#   independent binary heads that each read ONLY their own modality --
#     * video_head  reads the VISUAL branch only  -> P(video is fake)
#     * audio_head  reads the AUDIO  branch only  -> P(audio is fake)
#   This gives clean modality attribution ("which track is fake"), matching
#   FakeAVCeleb's 2x2 structure. The cross-attention path is kept ONLY as an
#   auxiliary "any-manipulation" regularizer -- it never feeds the two
#   attribution heads, so audio can't leak into the video decision (or v.v.).
#   Verdict = legit iff BOTH heads say real.
# =====================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs=40, num_channels=[64, 128, 256], kernel_size=3, dropout=0.3, target_seq_len=60):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size,
                          padding=(kernel_size-1)*dilation_size, dilation=dilation_size),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
        self.network = nn.Sequential(*layers)
        self.temporal_pool = nn.AdaptiveAvgPool1d(target_seq_len)
        self.proj = nn.Linear(num_channels[-1], 256)

    def forward(self, x):
        out = self.network(x)
        out = self.temporal_pool(out)
        out = out.transpose(1, 2)
        return self.proj(out)


class CrossFuseModel(nn.Module):
    """
    Two-head, separate-branch deepfake detector.
      forward() -> (video_logit, audio_logit, any_logit, attn_weights)
    video_logit / audio_logit are RAW logits (use BCEWithLogitsLoss to train,
    torch.sigmoid to read a probability). any_logit is the optional auxiliary
    "is anything manipulated" regularizer; it is NOT used for the final verdict.
    """
    def __init__(self, visual_dim=1792, audio_dim=256, hidden_dim=256, num_heads=4,
                 dropout=0.4, target_seq_len=60, use_aux=True):
        super(CrossFuseModel, self).__init__()
        self.use_aux = use_aux

        # ---- Visual branch (feeds video_head ONLY) ----
        self.visual_proj = nn.Linear(visual_dim, hidden_dim)
        self.visual_encoder = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, dim_feedforward=512,
            dropout=dropout, batch_first=True)

        # ---- Audio branch (feeds audio_head ONLY) ----
        self.audio_encoder = TemporalConvNet(num_inputs=40, num_channels=[64, 128, hidden_dim],
                                             dropout=dropout, target_seq_len=target_seq_len)

        # ---- Two independent binary heads ----
        self.video_head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1))
        self.audio_head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1))

        # ---- Optional auxiliary cross-modal regularizer (NOT in verdict path) ----
        if use_aux:
            self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads,
                                                         dropout=dropout, batch_first=True)
            self.layer_norm = nn.LayerNorm(hidden_dim)
            self.aux_head = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(64, 1))

    def forward(self, visual_features, audio_mfcc):
        # Visual branch -> video-only representation
        V = self.visual_proj(visual_features)          # (B, 60, hidden)
        V_enc = self.visual_encoder(V)                 # (B, 60, hidden)
        v_rep = V_enc.mean(dim=1)                       # (B, hidden)
        video_logit = self.video_head(v_rep).squeeze(-1)

        # Audio branch -> audio-only representation
        A = self.audio_encoder(audio_mfcc)             # (B, 60, hidden)
        a_rep = A.mean(dim=1)                           # (B, hidden)
        audio_logit = self.audio_head(a_rep).squeeze(-1)

        any_logit, attn_weights = None, None
        if self.use_aux:
            attn_out, attn_weights = self.cross_attention(query=V, key=A, value=A)
            fused = self.layer_norm(V + attn_out).mean(dim=1)
            any_logit = self.aux_head(fused).squeeze(-1)

        return video_logit, audio_logit, any_logit, attn_weights


# ---- Monte Carlo Dropout uncertainty utilities (now per-head) ----

def enable_mc_dropout(model):
    """
    Switch ONLY Dropout layers back on (BatchNorm stays in eval mode with
    frozen running stats). Call model.eval() first, then this, before
    predict_with_uncertainty().
    """
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

def predict_with_uncertainty(model, visual, audio, n_samples=20):
    """
    n_samples stochastic passes (dropout active). Returns FOUR arrays:
      mean_v, std_v : mean & std of P(video fake) across passes
      mean_a, std_a : mean & std of P(audio fake) across passes
    std_* is the MC-Dropout uncertainty for that head -- high std means the
    random subnetworks disagreed, i.e. the model is unsure about that modality.
    """
    model.eval()
    enable_mc_dropout(model)
    v_list, a_list = [], []
    with torch.no_grad():
        for _ in range(n_samples):
            v_logit, a_logit, _, _ = model(visual, audio)
            v_list.append(torch.sigmoid(v_logit).cpu().numpy())
            a_list.append(torch.sigmoid(a_logit).cpu().numpy())
    v_arr = np.stack(v_list, axis=0)
    a_arr = np.stack(a_list, axis=0)
    model.eval()  # restore full eval mode (dropout off) afterwards
    return v_arr.mean(axis=0), v_arr.std(axis=0), a_arr.mean(axis=0), a_arr.std(axis=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CrossFuseModel(visual_dim=1792, target_seq_len=60).to(device)
print("CrossFuse v3 architecture ready (two-head, separate branches). MC-Dropout utilities loaded.")


CrossFuse v3 architecture ready (two-head, separate branches). MC-Dropout utilities loaded.


### Cell 3 — Feature Extraction + Modality Labels + Synthetic Video Hard Negatives
**v3 change:** every saved sample now stores **two** labels — `video_label` (is the visual track fake?) and `audio_label` (is the audio track fake?) — derived from the FakeAVCeleb category. The old single `label` key is kept (`= video_fake OR audio_fake`) for backward compatibility, so DFDC tensors and any older `.pt` files still load.

- **Audio-desync negatives are dropped** — a time-shifted *real* audio track isn't "fake audio," so labeling it that way was misleading.
- **Self-blend negatives** map to `video_label=1, audio_label=0` (visual manipulation of a real clip, genuine audio).
- **Category-stratified fake selection** — because self-blend only feeds the video head, fakes are now pulled in balanced counts from each of RVFA / FVRA / FVFA so the **audio head actually gets fake-audio positives**. Watch the printed per-category counts: RVFA + FVFA are your audio-head positives.

**Before trusting anything downstream, check the printed identity-parsing sample against your actual Kaggle folder structure.** `get_identity_from_path()` assumes the standard `{category}/{ethnicity}/{gender}/{id}/{video}` layout.

**Important — delete cached features before rerunning:** Cell 3 skips any `.pt` that already exists, so if you have tensors from a previous run, `rm -rf /kaggle/working/extracted_features` first or the old (binary-labeled / desync) tensors will be silently reused. The self-blend landmark fix and the removal of ColorJitter/GaussianBlur from `img_transform` (v2.1) are retained.

In [ ]:
# =====================================================================
# CELL 3: FEATURE EXTRACTION + IDENTITY TAGGING + MODALITY LABELS +
#         SYNTHETIC VIDEO HARD NEGATIVES  (v3, two-head ready)
# WHAT CHANGED vs v2.1:
#   * Each sample now stores TWO labels: video_label (is the visual track
#     fake?) and audio_label (is the audio track fake?), derived from the
#     FakeAVCeleb category. The old single 'label' key is ALSO kept
#     (= video_fake OR audio_fake) for backward compatibility.
#   * Audio-desync synthetic negatives are DROPPED (a time-shifted real
#     audio track is not "fake audio" -- mislabeling it was misleading).
#   * Self-blend negatives now map to video_label=1, audio_label=0.
#   * Fake selection is now CATEGORY-STRATIFIED so the audio_head actually
#     gets fake-audio positives (RVFA/FVFA), which self-blend alone can't
#     provide.
#   * MFCC now passes through the shared cmvn_normalize() from Cell 1.
#   The dataset-discovery globs, MTCNN/EfficientNet extraction loop, the
#   60-frame sampling and the .pt skip-if-exists caching are UNCHANGED.
#   Already applied in v2.1 and kept: self-blend landmark fix + no
#   ColorJitter/GaussianBlur in img_transform.
# =====================================================================

import os
import glob
import cv2
import librosa
import random
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN

output_dir = "/kaggle/working/extracted_features"
os.makedirs(output_dir, exist_ok=True)

eff_net = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
eff_net.classifier = nn.Identity()
eff_net = eff_net.to(device).eval()

mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)

# Clean transform (no augmentation) -- identical to Cells 6 and 7 so that
# train / val / DFDC / live inference all share one preprocessing pipeline.
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_fakeavceleb_label(video_path: str) -> int:
    path_norm = video_path.replace("\\", "/").lower()
    if "realvideo-realaudio" in path_norm:
        return 0
    elif any(f in path_norm for f in ["realvideo-fakeaudio", "fakevideo-realaudio", "fakevideo-fakeaudio"]):
        return 1
    elif "real" in os.path.basename(path_norm) and "fake" not in os.path.basename(path_norm):
        return 0
    else:
        return 1

def get_fakeavceleb_category(video_path: str) -> str:
    path_norm = video_path.replace("\\", "/")
    for cat in ["RealVideo-RealAudio", "RealVideo-FakeAudio", "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]:
        if cat.lower() in path_norm.lower():
            return cat
    return "Unknown"

def category_to_modality_labels(category: str):
    """Return (video_label, audio_label) for a category string.
    video_label = 1 if the VISUAL track is manipulated.
    audio_label = 1 if the AUDIO track is manipulated."""
    if category == "SelfBlend-Synthetic":
        return 1, 0                      # visual manipulation only, real audio
    c = category.lower()
    video_fake = 1 if "fakevideo" in c else 0
    audio_fake = 1 if "fakeaudio" in c else 0
    return video_fake, audio_fake

def get_identity_from_path(video_path: str) -> str:
    """FakeAVCeleb layout: .../{category}/{ethnicity}/{gender}/{id}/{video}.mp4"""
    return os.path.basename(os.path.dirname(video_path))

def get_landmarks(pil_img, mtcnn_module):
    boxes, probs, landmarks = mtcnn_module.detect(pil_img, landmarks=True)
    if landmarks is None or len(landmarks) == 0:
        return None
    return landmarks[0]

def build_blend_mask(size, landmarks):
    w, h = size
    mask = np.zeros((h, w), dtype=np.uint8)
    if landmarks is not None:
        pts = landmarks.astype(np.int32)
        hull = cv2.convexHull(pts)
        cv2.fillConvexPoly(mask, hull, 255)
        kernel = np.ones((31, 31), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=2)
    else:
        cv2.ellipse(mask, (w // 2, h // 2), (w // 3, int(h * 0.4)), 0, 0, 360, 255, -1)
    mask = cv2.GaussianBlur(mask, (35, 35), 0)
    return mask

def self_blend_image(pil_img, landmarks):
    w, h = pil_img.size
    mask = build_blend_mask((w, h), landmarks).astype(np.float32)[..., None] / 255.0
    source = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)(pil_img)
    angle = random.uniform(-4, 4)
    tx, ty = random.uniform(-0.02, 0.02) * w, random.uniform(-0.02, 0.02) * h
    scale = random.uniform(0.97, 1.03)
    source = transforms.functional.affine(source, angle=angle, translate=(tx, ty), scale=scale, shear=0)
    base = np.array(pil_img).astype(np.float32)
    src = np.array(source).astype(np.float32)
    blended = mask * src + (1 - mask) * base
    return Image.fromarray(np.clip(blended, 0, 255).astype(np.uint8))

def extract_visual_embeddings(video_path, apply_self_blend=False):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
    step = max(1, total_frames // 60) if total_frames > 0 else max(1, fps // 10)

    count, vis_embs = 0, []
    while cap.isOpened() and len(vis_embs) < 60:
        ret, frame = cap.read()
        if not ret: break
        if count % step == 0:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            face = mtcnn(pil_img)
            if face is None:
                face_pil = pil_img.resize((224, 224))
            else:
                face_pil = transforms.ToPILImage()((face + 1.0) / 2.0)
            if apply_self_blend:
                # Landmarks detected on face_pil (the 224x224 crop we blend onto),
                # NOT the full frame -- otherwise the blend mask is empty.
                landmarks = get_landmarks(face_pil, mtcnn)
                face_pil = self_blend_image(face_pil, landmarks)
            face_tensor = img_transform(face_pil)
            with torch.no_grad():
                vis_embs.append(eff_net(face_tensor.unsqueeze(0).to(device)).squeeze(0).cpu())
        count += 1
    cap.release()
    if len(vis_embs) == 0:
        return None
    return torch.stack(vis_embs)

def extract_audio_mfcc(video_path):
    try:
        y, sr = librosa.load(video_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        if USE_MFCC_CMVN:
            mfcc = cmvn_normalize(mfcc)          # per-utterance normalization (Cell 1 helper)
        if mfcc.shape[1] < 128:
            mfcc = np.pad(mfcc, ((0, 0), (0, 128 - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :128]
        return mfcc.astype(np.float32)
    except Exception:
        return np.zeros((40, 128), dtype=np.float32)

def save_sample(name_suffix, video_path, visual, audio, category, synthetic):
    """Store both modality labels (derived from category) plus a backward-
    compatible overall 'label' = video_fake OR audio_fake."""
    identity = get_identity_from_path(video_path)
    video_label, audio_label = category_to_modality_labels(category)
    overall_label = int(video_label or audio_label)
    video_name = os.path.basename(video_path).split('.')[0] + name_suffix
    save_path = os.path.join(output_dir, f"{video_name}.pt")
    audio_tensor = audio if torch.is_tensor(audio) else torch.tensor(audio, dtype=torch.float32)
    torch.save({
        'visual': visual, 'audio': audio_tensor,
        'label': overall_label, 'video_label': video_label, 'audio_label': audio_label,
        'identity': identity, 'category': category, 'synthetic': synthetic
    }, save_path)

# 1. Locate the FakeAVCeleb pool
all_videos = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
if len(all_videos) == 0:
    all_videos = glob.glob("/kaggle/input/**/*.avi", recursive=True)

fav_videos = [v for v in all_videos if "dfdc" not in v.lower() and "deepfake-detection" not in v.lower()]

real_videos, fake_videos = [], []
for v in fav_videos:
    if get_fakeavceleb_label(v) == 0: real_videos.append(v)
    else: fake_videos.append(v)

print(f"Raw Candidate Pool: {len(real_videos)} Real | {len(fake_videos)} Fake")

random.shuffle(real_videos)
random.shuffle(fake_videos)

# --- Use ALL reals (pool is small ~178) ---
n_reals = len(real_videos)
selected_reals = real_videos[:n_reals]

# --- CATEGORY-STRATIFIED FAKE SELECTION (v3) -------------------------------
# The audio_head needs fake-AUDIO positives, which only exist in RVFA & FVFA.
# Self-blend synthetics only feed the video_head, so if we sliced fakes
# blindly the audio_head could end up with almost no positives. Pull a
# balanced count from EACH manipulated category instead.
fakes_by_cat = defaultdict(list)
for v in fake_videos:
    fakes_by_cat[get_fakeavceleb_category(v)].append(v)
for cat in fakes_by_cat:
    random.shuffle(fakes_by_cat[cat])

per_cat = 450  # explicit target: real is capped at ~500 (all of FakeAVCeleb's real identities),
               # so we scale the FAKE side instead. 500 real + 500 self-blend + 3*450 stratified
               # fakes = ~2350 total, landing in your target 2000-2500 range.
selected_fakes = []
for cat in ["RealVideo-FakeAudio", "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]:
    selected_fakes += fakes_by_cat.get(cat, [])[:per_cat]
random.shuffle(selected_fakes)

n_synth = n_reals if GENERATE_HARD_NEGATIVES else 0
print(f"Balance plan -> reals(RVRA): {len(selected_reals)} | original fakes (stratified): {len(selected_fakes)} "
      f"| self-blend video-fakes to add: {n_synth}")
cat_counts = {c: len(fakes_by_cat.get(c, [])[:per_cat]) for c in ['RealVideo-FakeAudio','FakeVideo-RealAudio','FakeVideo-FakeAudio']}
print(f"   per-category selected -> {cat_counts}  (RVFA + FVFA are your audio_head positives)")

if HELD_OUT_CATEGORY is not None:
    held_out_fakes = [v for v in selected_fakes if get_fakeavceleb_category(v) == HELD_OUT_CATEGORY]
    selected_fakes = [v for v in selected_fakes if get_fakeavceleb_category(v) != HELD_OUT_CATEGORY]
    print(f"Holding out {len(held_out_fakes)} '{HELD_OUT_CATEGORY}' videos entirely (cross-manipulation test)")

print("\nSample identity parsing (verify these look like real identity folders!):")
for v in (selected_reals[:3] + selected_fakes[:3]):
    vl, al = category_to_modality_labels(get_fakeavceleb_category(v))
    print(f"  {v}  ->  identity='{get_identity_from_path(v)}', category='{get_fakeavceleb_category(v)}', "
          f"video_label={vl}, audio_label={al}")

print(f"\nExtracting features. Real: {len(selected_reals)} | Fake: {len(selected_fakes)} "
      f"| Self-blend video-fakes per real: {1 if GENERATE_HARD_NEGATIVES else 0}")

extracted_count = 0
for video_path in tqdm(selected_reals + selected_fakes, desc="Extracting Tensors (v3)"):
    category = get_fakeavceleb_category(video_path)
    label = get_fakeavceleb_label(video_path)   # 0 for real (RVRA), used only to gate self-blend
    video_name = os.path.basename(video_path).split('.')[0]
    save_path = os.path.join(output_dir, f"{video_name}.pt")

    if not os.path.exists(save_path):
        visual = extract_visual_embeddings(video_path, apply_self_blend=False)
        if visual is not None:
            audio = extract_audio_mfcc(video_path)
            save_sample("", video_path, visual, audio, category=category, synthetic=False)
            extracted_count += 1

    # Self-blend hard negative: visual manipulation of a REAL video -> video_fake, audio real
    if GENERATE_HARD_NEGATIVES and label == 0:
        sb_path = os.path.join(output_dir, f"{video_name}_selfblend.pt")
        if not os.path.exists(sb_path):
            sb_visual = extract_visual_embeddings(video_path, apply_self_blend=True)
            if sb_visual is not None:
                sb_audio = extract_audio_mfcc(video_path)
                save_sample("_selfblend", video_path, sb_visual, sb_audio,
                            category="SelfBlend-Synthetic", synthetic=True)
                extracted_count += 1

print(f"\nExtraction complete! Total tensor files ready: {extracted_count}")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 151MB/s] 


Raw Candidate Pool: 500 Real | 21060 Fake
Balance plan -> reals(RVRA): 500 | original fakes (stratified): 1350 | self-blend video-fakes to add: 500
   per-category selected -> {'RealVideo-FakeAudio': 450, 'FakeVideo-RealAudio': 450, 'FakeVideo-FakeAudio': 450}  (RVFA + FVFA are your audio_head positives)

Sample identity parsing (verify these look like real identity folders!):
  /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Caucasian (European)/women/id00287/00005.mp4  ->  identity='id00287', category='RealVideo-RealAudio', video_label=0, audio_label=0
  /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/Asian (South)/women/id04530/00231.mp4  ->  identity='id04530', category='RealVideo-RealAudio', video_label=0, audio_label=0
  /kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id01179/00160.mp4  ->  identity='id01179', cate

Extracting Tensors (v3):   0%|          | 0/1850 [00:00<?, ?it/s]/tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000)
/tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000)
Extracting Tensors (v3):   0%|          | 1/1850 [00:23<12:12:36, 23.77s/it]/tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000)
/tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000)
Extracting Tensors (v3):   0%|          | 2/1850 [00:32<7:32:58, 14.71s/it] /tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=16000)
/tmp/ipykernel_58/3985271875.py:152: UserWarning: PySoundFile failed. Trying a

### Cell 4 — Identity-Disjoint Split + Two-Head Training
Identity-disjoint `GroupShuffleSplit` (unchanged — no person appears in both train and val; the leakage `assert` stays). **v3 change:** each sample now carries `(video_label, audio_label)`, the loss is `BCE(video_head) + BCE(audio_head)` plus a small auxiliary any-manipulation term, and we track a **separate val AUC for each head** — checkpointing on the mean of the two. Watch the **Audio AUC** in particular: it's the head most at risk of being data-starved, so if it stays near 0.5 while Video AUC rises, that's your signal the audio-fake pool (RVFA/FVFA counts printed in Cell 3) is too small. `drop_last=True` is set on the train loader so a size-1 final batch can't crash BatchNorm.

In [ ]:
# =====================================================================
# CELL 4: IDENTITY-DISJOINT SPLIT + TWO-HEAD TRAINING LOOP (v3)
# Multi-label: each sample carries (video_label, audio_label). Loss is
# BCE(video_head) + BCE(audio_head) (+ small auxiliary any-manipulation
# term). We track a SEPARATE val AUC for each head and checkpoint on the
# mean of the two.
# =====================================================================

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score

class CrossFuseDataset(Dataset):
    def __init__(self, folder, target_frames=60):
        self.files = glob.glob(os.path.join(folder, "*.pt"))
        self.target_frames = target_frames
        self.meta = [torch.load(f, weights_only=False) for f in self.files]

    def __len__(self):
        return len(self.files)

    def _labels(self, data):
        # Prefer the two modality labels; fall back to the legacy single
        # 'label' (e.g. DFDC tensors) so older files still load.
        if 'video_label' in data and 'audio_label' in data:
            return int(data['video_label']), int(data['audio_label'])
        lbl = int(data.get('label', 0))
        return lbl, lbl

    def __getitem__(self, idx):
        data = self.meta[idx]
        vis = data['visual']
        current_frames, feature_dim = vis.shape[0], vis.shape[1]
        if current_frames < self.target_frames:
            padding = torch.zeros((self.target_frames - current_frames, feature_dim), dtype=vis.dtype)
            vis = torch.cat([vis, padding], dim=0)
        else:
            vis = vis[:self.target_frames]
        vlab, alab = self._labels(data)
        return (vis, data['audio'],
                torch.tensor(vlab, dtype=torch.float32),
                torch.tensor(alab, dtype=torch.float32))

def safe_auc(y_true, y_score):
    """roc_auc_score errors if a batch/val split has only one class -> 0.5."""
    try:
        if len(set(int(v) for v in y_true)) < 2:
            return 0.5
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return 0.5

feature_dir = "/kaggle/working/extracted_features"
full_dataset = CrossFuseDataset(feature_dir, target_frames=60)

video_labels_all = [int(m.get('video_label', m.get('label', 0))) for m in full_dataset.meta]
audio_labels_all = [int(m.get('audio_label', m.get('label', 0))) for m in full_dataset.meta]
identities_all = [m['identity'] for m in full_dataset.meta]
synthetic_all = [m.get('synthetic', False) for m in full_dataset.meta]

print(f"Dataset -> Total: {len(full_dataset)}")
print(f"  Video head  -> real: {video_labels_all.count(0)} | fake: {video_labels_all.count(1)}")
print(f"  Audio head  -> real: {audio_labels_all.count(0)} | fake: {audio_labels_all.count(1)}  "
      f"(audio-fake count is the one to watch)")
print(f"  Synthetic self-blend hard negatives: {sum(synthetic_all)} | Unique identities: {len(set(identities_all))}")

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, val_idx = next(gss.split(np.zeros(len(full_dataset)), groups=identities_all))

train_ids = set(identities_all[i] for i in train_idx)
val_ids = set(identities_all[i] for i in val_idx)
overlap = train_ids & val_ids
print(f"Identities in train: {len(train_ids)} | Identities in val: {len(val_ids)} | Overlap: {len(overlap)} (must be 0)")
assert len(overlap) == 0, "Identity leakage detected in split -- check get_identity_from_path() in Cell 3!"

train_ds = Subset(full_dataset, train_idx)
val_ds = Subset(full_dataset, val_idx)

# drop_last=True so a size-1 final batch can't crash BatchNorm during training.
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

print(f"Split Complete -> Train: {len(train_ds)} samples | Validation: {len(val_ds)} samples")

model = CrossFuseModel(visual_dim=1792, target_seq_len=60, dropout=0.4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
AUX_WEIGHT = 0.5   # weight on the auxiliary any-manipulation regularizer

EPOCHS = 25
best_mean_auc = 0.0
best_val_auc = 0.0        # video-head AUC at the best checkpoint (used by Cell 6's gap)
best_val_audio_auc = 0.0
checkpoint_path = "/kaggle/working/crossfuse_fakeavceleb_best.pth"

print("\nStarting v3 Training Loop (identity-disjoint split + two heads)...\n")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for vis_batch, audio_batch, vlab_batch, alab_batch in train_loader:
        vis_batch, audio_batch = vis_batch.to(device), audio_batch.to(device)
        vlab_batch, alab_batch = vlab_batch.to(device), alab_batch.to(device)
        optimizer.zero_grad()
        video_logit, audio_logit, any_logit, _ = model(vis_batch, audio_batch)
        loss = criterion(video_logit, vlab_batch) + criterion(audio_logit, alab_batch)
        if any_logit is not None:
            any_lab = torch.clamp(vlab_batch + alab_batch, 0, 1)
            loss = loss + AUX_WEIGHT * criterion(any_logit, any_lab)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / max(1, len(train_loader))

    model.eval()
    v_true, v_prob, a_true, a_prob = [], [], [], []
    with torch.no_grad():
        for vis_batch, audio_batch, vlab_batch, alab_batch in val_loader:
            vis_batch, audio_batch = vis_batch.to(device), audio_batch.to(device)
            video_logit, audio_logit, _, _ = model(vis_batch, audio_batch)
            v_prob.extend(torch.sigmoid(video_logit).cpu().numpy())
            a_prob.extend(torch.sigmoid(audio_logit).cpu().numpy())
            v_true.extend(vlab_batch.numpy())
            a_true.extend(alab_batch.numpy())

    video_auc = safe_auc(v_true, v_prob)
    audio_auc = safe_auc(a_true, a_prob)
    mean_auc = 0.5 * (video_auc + audio_auc)

    if mean_auc > best_mean_auc:
        best_mean_auc = mean_auc
        best_val_auc = video_auc
        best_val_audio_auc = audio_auc
        torch.save(model.state_dict(), checkpoint_path)
        star = "★ (New Best)"
    else:
        star = ""

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> Train Loss: {avg_train_loss:.4f} | "
          f"Video AUC: {video_auc:.4f} | Audio AUC: {audio_auc:.4f} | Mean: {mean_auc:.4f} {star}")

print(f"\nTraining Complete! Best checkpoint saved to: {checkpoint_path}")
print(f"Best -> Video AUC: {best_val_auc:.4f} | Audio AUC: {best_val_audio_auc:.4f} | Mean: {best_mean_auc:.4f}")
print("A lower Video AUC than the old randomly-split 89% is expected and is an honest ablation result.")

### Cell 5 — Per-Head Calibration + Uncertainty Report
**v3 change:** calibration and uncertainty are now computed **per head**. Two Youden's-J thresholds are found and saved (`optimal_threshold_video.npy`, `optimal_threshold_audio.npy`; the legacy `optimal_threshold.npy` is kept pointing at the video threshold). You get a per-head classification report, per-head ECE, per-head MC-Dropout uncertainty, a 2×3 grid of confusion/ROC/reliability plots (video row, audio row), and — the new attribution evidence — a **4-way confusion matrix over the derived categories** (RVRA/RVFA/FVRA/FVFA) built from the two head decisions, showing how well the model pins down *which* modality is fake.

In [ ]:
# =====================================================================
# CELL 5: PER-HEAD THRESHOLD CALIBRATION + UNCERTAINTY / ECE REPORT (v3)
# Two heads -> two Youden's-J thresholds (thr_video, thr_audio), two ECEs,
# two reliability diagrams, per-head MC-Dropout uncertainty, PLUS a 4-way
# confusion matrix over the derived categories (RVRA/RVFA/FVRA/FVFA) that
# shows how well the model ATTRIBUTES which modality is fake.
# =====================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report

eval_model = CrossFuseModel(visual_dim=1792, target_seq_len=60).to(device)
eval_model.load_state_dict(torch.load("/kaggle/working/crossfuse_fakeavceleb_best.pth", map_location=device, weights_only=False))
eval_model.eval()

v_true, v_prob, v_unc = [], [], []
a_true, a_prob, a_unc = [], [], []
with torch.no_grad():
    for vis_batch, audio_batch, vlab_batch, alab_batch in val_loader:
        vis_batch, audio_batch = vis_batch.to(device), audio_batch.to(device)
        mean_v, std_v, mean_a, std_a = predict_with_uncertainty(eval_model, vis_batch, audio_batch, n_samples=N_MC_SAMPLES)
        v_prob.extend(mean_v); v_unc.extend(std_v); v_true.extend(vlab_batch.numpy())
        a_prob.extend(mean_a); a_unc.extend(std_a); a_true.extend(alab_batch.numpy())

v_true, v_prob, v_unc = np.array(v_true, dtype=int), np.array(v_prob), np.array(v_unc)
a_true, a_prob, a_unc = np.array(a_true, dtype=int), np.array(a_prob), np.array(a_unc)

def youden_threshold(y_true, y_prob, default=0.5):
    if len(set(int(v) for v in y_true)) < 2:
        return default
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    return float(thr[np.argmax(tpr - fpr)])

def expected_calibration_error(probs, labels, n_bins=5):
    """Gap between confidence and accuracy across confidence bins (lower better)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece, bin_accs, bin_confs = 0.0, [], []
    for i in range(n_bins):
        lo = bins[i] if i > 0 else -0.01   # include prob==0 in the first bin
        mask = (probs > lo) & (probs <= bins[i + 1])
        if mask.sum() > 0:
            bin_acc = (labels[mask] == (probs[mask] >= 0.5)).mean()
            bin_conf = probs[mask].mean()
            ece += (mask.sum() / len(probs)) * abs(bin_acc - bin_conf)
            bin_accs.append(bin_acc); bin_confs.append(bin_conf)
        else:
            bin_accs.append(np.nan); bin_confs.append(np.nan)
    return ece, bin_accs, bin_confs

thr_video = youden_threshold(v_true, v_prob)
thr_audio = youden_threshold(a_true, a_prob)
np.save("/kaggle/working/optimal_threshold_video.npy", np.array([thr_video]))
np.save("/kaggle/working/optimal_threshold_audio.npy", np.array([thr_audio]))
np.save("/kaggle/working/optimal_threshold.npy", np.array([thr_video]))  # legacy alias

v_pred = (v_prob >= thr_video).astype(int)
a_pred = (a_prob >= thr_audio).astype(int)

print("\n" + "="*60)
print(f"  CROSSFUSE v3 VALIDATION REPORT")
print(f"  Video threshold: {thr_video:.4f} | Audio threshold: {thr_audio:.4f}")
print("="*60)
print("\n--- VIDEO HEAD (is the visual track fake?) ---")
print(classification_report(v_true, v_pred, target_names=["Real vid", "Fake vid"], zero_division=0))
print(f"Mean MC-Dropout uncertainty (video): {v_unc.mean():.4f}")
print("\n--- AUDIO HEAD (is the audio track fake?) ---")
print(classification_report(a_true, a_pred, target_names=["Real aud", "Fake aud"], zero_division=0))
print(f"Mean MC-Dropout uncertainty (audio): {a_unc.mean():.4f}")

ece_v, bacc_v, bconf_v = expected_calibration_error(v_prob, v_true)
ece_a, bacc_a, bconf_a = expected_calibration_error(a_prob, a_true)
print(f"\nExpected Calibration Error -> Video: {ece_v:.4f} | Audio: {ece_a:.4f}  (lower is better)")

# ---- Per-head plots: 2 rows (video, audio) x 3 cols (confusion, ROC, reliability) ----
sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (name, yt, yp, pred, thr, ece, bacc, bconf) in enumerate([
        ("Video", v_true, v_prob, v_pred, thr_video, ece_v, bacc_v, bconf_v),
        ("Audio", a_true, a_prob, a_pred, thr_audio, ece_a, bacc_a, bconf_a)]):
    cm = confusion_matrix(yt, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[row, 0],
                xticklabels=[f"Pred real", f"Pred fake"],
                yticklabels=[f"Actual real", f"Actual fake"], annot_kws={"size": 14})
    axes[row, 0].set_title(f"{name} Head — Confusion (p≥{thr:.2f})", fontsize=12, fontweight='bold')

    if len(set(int(v) for v in yt)) > 1:
        fpr, tpr, thrs = roc_curve(yt, yp)
        roc_auc = auc(fpr, tpr)
        oi = np.argmax(tpr - fpr)
        axes[row, 1].plot(fpr, tpr, color='#1f77b4', lw=3, label=f'{name} (AUC={roc_auc:.3f})')
        axes[row, 1].scatter(fpr[oi], tpr[oi], color='red', s=90, zorder=5, label=f'Cutoff {thr:.2f}')
    axes[row, 1].plot([0, 1], [0, 1], color='gray', lw=1.2, linestyle='--')
    axes[row, 1].set_xlabel('FPR'); axes[row, 1].set_ylabel('TPR')
    axes[row, 1].set_title(f'{name} Head — ROC', fontsize=12, fontweight='bold')
    axes[row, 1].legend(loc="lower right")

    axes[row, 2].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    axes[row, 2].plot(bconf, bacc, marker='o', color='#d62728', label=f'{name} (ECE={ece:.3f})')
    axes[row, 2].set_xlabel('Mean predicted confidence'); axes[row, 2].set_ylabel('Empirical accuracy')
    axes[row, 2].set_title(f'{name} Head — Reliability', fontsize=12, fontweight='bold')
    axes[row, 2].legend(loc="upper left")

plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v3_perhead_plots.png", dpi=200, bbox_inches='tight')
plt.show()

# ---- 4-way attribution confusion (derived category from the two heads) ----
def to_category_idx(vf, af):
    return {(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3}[(int(vf), int(af))]

cat_names = ["RVRA\n(real/real)", "RVFA\n(real/fakeAud)", "FVRA\n(fakeVid/real)", "FVFA\n(fake/fake)"]
true_cat = [to_category_idx(vt, at) for vt, at in zip(v_true, a_true)]
pred_cat = [to_category_idx(vp, ap) for vp, ap in zip(v_pred, a_pred)]
cm4 = confusion_matrix(true_cat, pred_cat, labels=[0, 1, 2, 3])

plt.figure(figsize=(7.5, 6))
sns.heatmap(cm4, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=cat_names, yticklabels=cat_names, annot_kws={"size": 12})
plt.title("Modality Attribution — Derived 4-Category Confusion", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("/kaggle/working/crossfuse_v3_attribution_matrix.png", dpi=200, bbox_inches='tight')
plt.show()

print(f"\nThresholds saved -> video: {thr_video:.4f}, audio: {thr_audio:.4f}")

### Cell 6 — DFDC Zero-Shot Benchmark (video head)
DFDC is a **video** benchmark (per-clip real/fake, audio generally genuine), so the primary zero-shot number is the **video head** evaluated against the DFDC label; the generalization gap is `best video-head val AUC − DFDC video-head AUC`. A secondary "combined verdict" row (fake if *either* head fires) is also reported. The two DFDC bug fixes from v2 are retained — labels carried as `(path, label)` tuples, and every detected face routed through the same `img_transform` normalization used in Cells 3 and 7 — and the same MFCC CMVN is applied so DFDC audio matches training. DFDC tensors store `video_label = label, audio_label = 0`, so `CrossFuseDataset` loads them without change.

In [ ]:
# =====================================================================
# CELL 6: CALIBRATED DFDC ZERO-SHOT BENCHMARK (v3)
# DFDC is a VIDEO benchmark (per-clip real/fake, audio usually genuine),
# so the primary zero-shot number is the VIDEO head evaluated against the
# DFDC label. We also report a secondary "combined verdict" row (fake if
# EITHER head fires). Bug-1 (label bound to path) and Bug-2 (face routed
# through the same normalization) fixes from v2 are retained.
# =====================================================================

import json
import os
import glob
import cv2
import librosa
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

dfdc_output_dir = "/kaggle/working/dfdc_test_features"
os.makedirs(dfdc_output_dir, exist_ok=True)

eff_net = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
eff_net.classifier = nn.Identity()
eff_net = eff_net.to(device).eval()

mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_dfdc_ground_truth(max_per_class=100):
    """Bug-1 fix: returns (path, label) tuples so the label always travels
    WITH its path through every later shuffle/slice."""
    labeled = []
    meta_files = glob.glob("/kaggle/input/**/metadata.json", recursive=True)
    for meta_path in meta_files:
        folder = os.path.dirname(meta_path)
        with open(meta_path, 'r') as f:
            data = json.load(f)
        for vid_name, info in data.items():
            full_path = os.path.join(folder, vid_name)
            if not os.path.exists(full_path):
                continue
            lbl = info.get('label', '').upper()
            if lbl == 'REAL':
                labeled.append((full_path, 0))
            elif lbl == 'FAKE':
                labeled.append((full_path, 1))

    reals = [x for x in labeled if x[1] == 0]
    fakes = [x for x in labeled if x[1] == 1]
    random.shuffle(reals)
    random.shuffle(fakes)
    limit = min(max_per_class, len(reals), len(fakes))
    selected = reals[:limit] + fakes[:limit]
    random.shuffle(selected)   # safe -- label is bound to path inside the tuple
    return selected

selected_dfdc = load_dfdc_ground_truth(max_per_class=100)
print(f"Extracting 60-frame features for {len(selected_dfdc)} unseen DFDC videos "
      f"({sum(1 for _, l in selected_dfdc if l==0)} real / {sum(1 for _, l in selected_dfdc if l==1)} fake)...")

for video_path, label in tqdm(selected_dfdc, desc="Extracting DFDC Tensors (v3)"):
    video_name = "dfdc_" + os.path.basename(video_path).split('.')[0]
    save_path = os.path.join(dfdc_output_dir, f"{video_name}.pt")
    if os.path.exists(save_path):
        continue

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
    step = max(1, total_frames // 60) if total_frames > 0 else max(1, fps // 10)

    count, vis_embs = 0, []
    while cap.isOpened() and len(vis_embs) < 60:
        ret, frame = cap.read()
        if not ret:
            break
        if count % step == 0:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            face = mtcnn(pil_img)
            # Bug-2 fix: always round-trip a detected face through the same
            # PIL -> img_transform normalization used in Cells 3 and 7.
            if face is None:
                face_pil = pil_img.resize((224, 224))
            else:
                face_pil = transforms.ToPILImage()((face + 1.0) / 2.0)
            face_tensor = img_transform(face_pil)
            with torch.no_grad():
                vis_embs.append(eff_net(face_tensor.unsqueeze(0).to(device)).squeeze(0).cpu())
        count += 1
    cap.release()

    if len(vis_embs) == 0:
        continue
    vis_tensor = torch.stack(vis_embs)

    try:
        y, sr = librosa.load(video_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        if USE_MFCC_CMVN:
            mfcc = cmvn_normalize(mfcc)                # same normalization as Cells 3 and 7
        if mfcc.shape[1] < 128:
            mfcc = np.pad(mfcc, ((0, 0), (0, 128 - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :128]
        audio_tensor = torch.tensor(mfcc, dtype=torch.float32)
    except Exception:
        audio_tensor = torch.zeros((40, 128), dtype=torch.float32)

    # DFDC only has a per-clip video label -> store it as both 'label' and
    # 'video_label'; 'audio_label'=0 (DFDC audio is generally genuine).
    torch.save({'visual': vis_tensor, 'audio': audio_tensor,
                'label': label, 'video_label': label, 'audio_label': 0}, save_path)

dfdc_dataset = CrossFuseDataset(dfdc_output_dir, target_frames=60)
dfdc_loader = DataLoader(dfdc_dataset, batch_size=16, shuffle=False)

eval_model = CrossFuseModel(visual_dim=1792, target_seq_len=60).to(device)
eval_model.load_state_dict(torch.load("/kaggle/working/crossfuse_fakeavceleb_best.pth", map_location=device, weights_only=False))
eval_model.eval()

def _load_thr(path, default):
    return float(np.load(path)[0]) if os.path.exists(path) else default
thr_video = _load_thr("/kaggle/working/optimal_threshold_video.npy", 0.5)
thr_audio = _load_thr("/kaggle/working/optimal_threshold_audio.npy", 0.5)

dfdc_labels, dfdc_vprob, dfdc_aprob, dfdc_vunc = [], [], [], []
with torch.no_grad():
    for vis_batch, audio_batch, vlab_batch, alab_batch in dfdc_loader:
        vis_batch, audio_batch = vis_batch.to(device), audio_batch.to(device)
        mean_v, std_v, mean_a, std_a = predict_with_uncertainty(eval_model, vis_batch, audio_batch, n_samples=N_MC_SAMPLES)
        dfdc_labels.extend(vlab_batch.numpy())   # DFDC ground truth == video label
        dfdc_vprob.extend(mean_v)
        dfdc_aprob.extend(mean_a)
        dfdc_vunc.extend(std_v)

dfdc_labels = np.array(dfdc_labels, dtype=int)
dfdc_vprob, dfdc_aprob = np.array(dfdc_vprob), np.array(dfdc_aprob)

# Primary: video head vs DFDC label
video_preds = (dfdc_vprob >= thr_video).astype(int)
# Secondary: combined verdict (fake if EITHER head fires)
combined_preds = ((dfdc_vprob >= thr_video) | (dfdc_aprob >= thr_audio)).astype(int)

print("\n" + "="*60)
print(f"  UNSEEN DFDC REPORT -- v3 (VIDEO head, threshold {thr_video:.2f})")
print("="*60)
print(classification_report(dfdc_labels, video_preds, target_names=["Real (0)", "Fake (1)"], zero_division=0))

dfdc_video_acc = accuracy_score(dfdc_labels, video_preds)
dfdc_video_auc = safe_auc(dfdc_labels, dfdc_vprob)
dfdc_ece, _, _ = expected_calibration_error(dfdc_vprob, dfdc_labels)
combined_acc = accuracy_score(dfdc_labels, combined_preds)

print(f">> Video-head Zero-Shot Accuracy on DFDC : {dfdc_video_acc*100:.2f}%")
print(f">> Video-head Zero-Shot AUC-ROC on DFDC  : {dfdc_video_auc:.4f}")
print(f">> Video-head Zero-Shot ECE on DFDC      : {dfdc_ece:.4f}")
print(f">> Mean MC-Dropout uncertainty (video)   : {np.mean(dfdc_vunc):.4f}")
print(f">> [secondary] Combined-verdict accuracy : {combined_acc*100:.2f}% "
      f"(fake if video OR audio head fires)")

generalization_gap = best_val_auc - dfdc_video_auc
print(f"\n>> CROSS-DATASET GENERALIZATION GAP (Val Video AUC - DFDC Video AUC): {generalization_gap:.4f}")
print("   Headline Table 3 number -- compare against the same gap with")
print("   GENERATE_HARD_NEGATIVES=False to show what the synthetic hard")
print("   negatives bought you.")

plt.figure(figsize=(6, 5))
cm = confusion_matrix(dfdc_labels, video_preds, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=["Pred Real", "Pred Fake"], yticklabels=["Actual Real", "Actual Fake"], annot_kws={"size": 14})
plt.title(f"DFDC Generalization (Video head, AUC = {dfdc_video_auc:.4f})", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("/kaggle/working/dfdc_generalization_matrix_v3.png", dpi=200)
plt.show()

### Cell 7 — Interactive Inference (per-modality, uncertainty-aware)
Same upload-and-analyze workflow, but the verdict is now **per modality**. The engine reports `P(video fake)` and `P(audio fake)` separately, each with its own MC-Dropout uncertainty and its own calibrated threshold, then combines them:
- both real → **AUTHENTIC**
- video only → **DEEPFAKE — video track manipulated**
- audio only → **DEEPFAKE — audio track manipulated**
- both → **DEEPFAKE — video + audio both manipulated**
- either head too uncertain → **flagged for human review**

This is the payoff of the two-head design: for an uploaded clip you now find out *which* track is fake, not just a single blurred real/fake score.

In [ ]:
# =====================================================================
# CELL 7: INTERACTIVE IN-THE-WILD INFERENCE -- PER-MODALITY VERDICT (v3)
# Reports P(video fake) and P(audio fake) separately, each with its own
# MC-Dropout uncertainty, then combines: LEGIT iff both heads say real,
# otherwise names which modality (or both) is manipulated. If either head
# is too uncertain, the clip is flagged for human review instead.
# =====================================================================

import os
import io
import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from facenet_pytorch import MTCNN

UNCERTAINTY_FLAG_THRESHOLD = 0.15  # tune against the val uncertainty distribution (Cell 5)

def analyze_custom_video(video_path: str, model_checkpoint: str = "/kaggle/working/crossfuse_fakeavceleb_best.pth"):
    print("\n" + "="*65)
    print("      CROSSFUSE v3 IN-THE-WILD FORENSIC INFERENCE ENGINE")
    print("="*65)

    if not os.path.exists(video_path):
        raise FileNotFoundError(f"ERROR: Could not find video at '{video_path}'.")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[*] Target Video      : {os.path.basename(video_path)}")
    print(f"[*] Hardware Device   : {device.type.upper()}")

    print("[*] Loading visual backbone (MTCNN + EfficientNet-B4)...")
    mtcnn = MTCNN(image_size=224, margin=20, keep_all=False, device=device)
    eff_net = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
    eff_net.classifier = nn.Identity()
    eff_net = eff_net.to(device).eval()

    img_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("[*] Sampling up to 60 facial embeddings...")
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = max(1, int(cap.get(cv2.CAP_PROP_FPS)))
    step = max(1, total_frames // 60) if total_frames > 0 else max(1, fps // 10)

    count, vis_embs = 0, []
    while cap.isOpened() and len(vis_embs) < 60:
        ret, frame = cap.read()
        if not ret: break
        if count % step == 0:
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            face = mtcnn(pil_img)
            if face is None:
                face_tensor = img_transform(pil_img.resize((224, 224)))
            else:
                face_pil = transforms.ToPILImage()((face + 1.0) / 2.0)
                face_tensor = img_transform(face_pil)
            with torch.no_grad():
                vis_embs.append(eff_net(face_tensor.unsqueeze(0).to(device)).squeeze(0).cpu())
        count += 1
    cap.release()

    if len(vis_embs) == 0:
        raise RuntimeError("ERROR: Could not extract any valid face crops from the video!")

    vis_tensor = torch.stack(vis_embs)
    if vis_tensor.shape[0] < 60:
        padding = torch.zeros((60 - vis_tensor.shape[0], 1792), dtype=vis_tensor.dtype)
        vis_tensor = torch.cat([vis_tensor, padding], dim=0)
    else:
        vis_tensor = vis_tensor[:60]

    print("[*] Extracting acoustic speech MFCCs...")
    try:
        y, sr = librosa.load(video_path, sr=16000)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
        if USE_MFCC_CMVN:
            mfcc = cmvn_normalize(mfcc)                 # same normalization as Cells 3 and 6
        if mfcc.shape[1] < 128:
            mfcc = np.pad(mfcc, ((0, 0), (0, 128 - mfcc.shape[1])))
        else:
            mfcc = mfcc[:, :128]
        aud_tensor = torch.tensor(mfcc, dtype=torch.float32)
    except Exception as e:
        print(f"[!] WARNING: Audio extraction fallback used ({e}).")
        aud_tensor = torch.zeros((40, 128), dtype=torch.float32)

    print("[*] Running two-head inference with MC-Dropout uncertainty...")
    model = CrossFuseModel(visual_dim=1792, target_seq_len=60).to(device)
    model.load_state_dict(torch.load(model_checkpoint, map_location=device, weights_only=False))

    def _load_thr(path, default):
        return float(np.load(path)[0]) if os.path.exists(path) else default
    thr_video = _load_thr("/kaggle/working/optimal_threshold_video.npy", 0.5)
    thr_audio = _load_thr("/kaggle/working/optimal_threshold_audio.npy", 0.5)

    vis_input = vis_tensor.unsqueeze(0).to(device)
    aud_input = aud_tensor.unsqueeze(0).to(device)

    mean_v, std_v, mean_a, std_a = predict_with_uncertainty(model, vis_input, aud_input, n_samples=N_MC_SAMPLES)
    p_video_fake = float(mean_v[0]); unc_video = float(std_v[0])
    p_audio_fake = float(mean_a[0]); unc_audio = float(std_a[0])

    video_is_fake = p_video_fake >= thr_video
    audio_is_fake = p_audio_fake >= thr_audio
    video_uncertain = unc_video >= UNCERTAINTY_FLAG_THRESHOLD
    audio_uncertain = unc_audio >= UNCERTAINTY_FLAG_THRESHOLD

    if video_uncertain or audio_uncertain:
        which = []
        if video_uncertain: which.append("video")
        if audio_uncertain: which.append("audio")
        verdict = f"UNCERTAIN ({'/'.join(which)}) -- flagged for human review"
    elif video_is_fake and audio_is_fake:
        verdict = "DEEPFAKE -- video + audio BOTH manipulated"
    elif video_is_fake:
        verdict = "DEEPFAKE -- VIDEO track manipulated (audio looks real)"
    elif audio_is_fake:
        verdict = "DEEPFAKE -- AUDIO track manipulated (video looks real)"
    else:
        verdict = "AUTHENTIC MEDIA (video and audio both real)"

    print("\n" + "-"*65)
    print("                     FORENSIC ANALYSIS REPORT (v3)")
    print("-"*65)
    print(f"  > P(video fake)   : {p_video_fake*100:6.2f}%   (threshold {thr_video*100:.1f}%, uncertainty {unc_video:.4f})")
    print(f"  > P(audio fake)   : {p_audio_fake*100:6.2f}%   (threshold {thr_audio*100:.1f}%, uncertainty {unc_audio:.4f})")
    print(f"  > Uncertainty flag: {UNCERTAINTY_FLAG_THRESHOLD}")
    print("-"*65)
    print(f"  > VIDEO verdict   : {'FAKE' if video_is_fake else 'REAL'}")
    print(f"  > AUDIO verdict   : {'FAKE' if audio_is_fake else 'REAL'}")
    print(f"  > FINAL VERDICT   : {verdict}")
    print("="*65 + "\n")

upload_widget = widgets.FileUpload(accept='.mp4,.avi,.mov', multiple=False, description='Upload Video', button_style='primary')
output_area = widgets.Output()

def on_file_upload(change):
    with output_area:
        clear_output()
        if not upload_widget.value:
            return
        print("Processing uploaded file...")
        uploaded_data = upload_widget.value
        if isinstance(uploaded_data, (tuple, list)):
            file_info = uploaded_data[0]
            file_name = file_info['name']
            content = file_info['content']
        elif isinstance(uploaded_data, dict):
            file_name = list(uploaded_data.keys())[0]
            content = uploaded_data[file_name]['content']

        save_path = os.path.join("/kaggle/working", f"interactive_{file_name}")
        if isinstance(content, memoryview):
            content = content.tobytes()
        with open(save_path, "wb") as f:
            f.write(content)
        print(f"File saved cleanly to: {save_path}")
        analyze_custom_video(save_path)

upload_widget.observe(on_file_upload, names='value')
print("Click the button below to upload your test video directly:")
display(widgets.VBox([upload_widget, output_area]))